# Vector store test
Loads the FAISS index and runs a test query against the real-data chunks.

In [1]:
from pathlib import Path
import sys

# Works on every machine: find the project root (the folder that contains rag/pipeline.py)
PROJECT_ROOT = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "rag" / "pipeline.py").exists()
)
RAG_DIR = PROJECT_ROOT / "rag"
import faiss
import numpy as np


# ============================================================
# Configuration
# ============================================================

INDEX_FILE = RAG_DIR / "vector_store" / "processed" / "vector_store" / "faiss.index"


# ============================================================
# FAISS Vector Store
# ============================================================

class FAISSVectorStore:

    def __init__(self, index_path=INDEX_FILE):

        self.index_path = Path(
            index_path
        )

        self.index = None

        self.load()


    # ========================================================
    # Load FAISS index
    # ========================================================

    def load(self):

        if not self.index_path.exists():

            raise FileNotFoundError(
                f"FAISS index not found: "
                f"{self.index_path}"
            )

        self.index = faiss.read_index(
            str(self.index_path)
        )

        print(
            f"FAISS index loaded."
        )

        print(
            f"Vectors: {self.index.ntotal}"
        )

        print(
            f"Dimension: {self.index.d}"
        )


    # ========================================================
    # Search
    # ========================================================

    def search(
        self,
        query_embedding,
        top_k=5
    ):

        # Make sure shape is:
        # (1, embedding_dimension)

        query_embedding = np.asarray(
            query_embedding,
            dtype=np.float32
        )

        if query_embedding.ndim == 1:

            query_embedding = (
                query_embedding.reshape(1, -1)
            )

        # Search
        scores, indices = self.index.search(
            query_embedding,
            top_k
        )

        results = []

        for score, index in zip(
            scores[0],
            indices[0]
        ):

            results.append({
                "chunk_id": int(index),
                "score": float(score)
            })

        return results

In [2]:
print("Exists:", INDEX_FILE.exists())
print("Path:", INDEX_FILE)
assert INDEX_FILE.exists(), "run index_builder.ipynb first"

Exists: True
Path: C:\Users\Ahmed\.zcode\workspace\default\Nasa Rag SYS\rag\vector_store\processed\vector_store\faiss.index


In [3]:
from sentence_transformers import SentenceTransformer


# ============================================================
# Load BGE embedding model
# ============================================================

model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)


# ============================================================
# Load FAISS Vector Store
# ============================================================

vector_store = FAISSVectorStore()


# ============================================================
# Test Query
# ============================================================

query = (
    "How does oxygen concentration affect "
    "the burn time of butane flames in microgravity?"
)

print("\nQuery:")
print(query)


# ============================================================
# Create Query Embedding
# ============================================================

query_embedding = model.encode(
    query,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print("\nQuery embedding shape:")
print(query_embedding.shape)


# ============================================================
# FAISS Search
# ============================================================

results = vector_store.search(
    query_embedding,
    top_k=5
)


# ============================================================
# Display Results
# ============================================================

print("\n================================")
print("FAISS SEARCH RESULTS")
print("================================")

for rank, result in enumerate(
    results,
    start=1
):

    print(
        f"Rank {rank} | "
        f"Chunk ID: {result['chunk_id']} | "
        f"Score: {result['score']:.4f}"
    )

C:\Users\Ahmed\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  43%|████▎     | 86/199 [00:00<00:00, 837.05it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1698.04it/s]

FAISS index loaded.
Vectors: 771
Dimension: 384

Query:
How does oxygen concentration affect the burn time of butane flames in microgravity?

Query embedding shape:
(384,)

FAISS SEARCH RESULTS
Rank 1 | Chunk ID: 730 | Score: 0.8201
Rank 2 | Chunk ID: 726 | Score: 0.8093
Rank 3 | Chunk ID: 766 | Score: 0.8038
Rank 4 | Chunk ID: 731 | Score: 0.7969
Rank 5 | Chunk ID: 716 | Score: 0.7935


In [4]:
import os
import sys

print("Current working directory:")
print(os.getcwd())

print("\nPython path:")
for path in sys.path:
    print(path)

Current working directory:
C:\Users\Ahmed\.zcode\workspace\default\Nasa Rag SYS\rag\vector_store

Python path:
C:\Users\Ahmed\AppData\Local\Programs\Python\Python313\python313.zip
C:\Users\Ahmed\AppData\Local\Programs\Python\Python313\DLLs
C:\Users\Ahmed\AppData\Local\Programs\Python\Python313\Lib
C:\Users\Ahmed\AppData\Local\Programs\Python\Python313

C:\Users\Ahmed\AppData\Local\Programs\Python\Python313\Lib\site-packages


In [5]:
VECTOR_STORE_DIR = str(RAG_DIR / "vector_store")

if VECTOR_STORE_DIR not in sys.path:
    sys.path.insert(0, VECTOR_STORE_DIR)

print("Vector store directory added.")

Vector store directory added.


In [6]:
from Metadata_filter import MetadataFilter

print("MetadataFilter imported successfully!")

MetadataFilter imported successfully!


In [7]:
chunks_file = RAG_DIR / "embeddings" / "processed" / "chunks" / "chunks.json"

metadata_filter = MetadataFilter(
    chunks_file
)

enriched_results = metadata_filter.enrich_results(
    results
)
assert all(r["metadata"]["experiment_id"].startswith(("CFI-", "SUMMARY-", "PSI-DOC-")) for r in enriched_results)

Loaded 771 chunks.


In [8]:
for rank, result in enumerate(enriched_results, start=1):

    print("\n" + "=" * 70)
    print(f"Rank: {rank}")
    print(f"Chunk ID: {result['chunk_id']}")
    print(f"Score: {result['score']:.4f}")

    print("\nMetadata:")
    print(result["metadata"])

    print("\nText:")
    print(result["text"])


Rank: 1
Chunk ID: 730
Score: 0.8201

Metadata:
{'experiment_id': 'PSI-DOC-inv-PSI-25', 'doc_type': 'nasa_psi', 'psi_source_type': 'investigation', 'investigation_id': 'PSI-25', 'source_url': 'https://psi.nasa.gov/physci/repo/data/investigations/PSI-25', 'doi': '10.60555/4qc4-de67', 'chunk_type': 'nasa_psi_document', 'source': 'PSI-DOC-inv-PSI-25.md'}

Text:
# BASS-II: Burning and Suppression of Solids - II

## Summary

BASS-II (PSI-25): Burning and Suppression of Solids - II. Platform: International Space Station (ISS). Hardware: Microgravity Science Glovebox (MSG). Objective: BASS-II results contribute to the combustion computational models used in the design of fire detection and suppression systems in microgravity and on Earth. Objectives Include: 1) Study the ignition, flame growth, flame spread, and extinction limits for solid fuels burning in low-velocity forced flows in microgravity. 2) Begin to bridge the gap between the normal gravity NASA-STD-6001 Test #1 method, ground-base